# Commentary Ingestion Pipeline v3
## Key change from v2: local YouTube cache replaces per-race API searches

**Quota usage comparison**
- v2: 4 channels × 100 units × 90 races = **36,000 units/day** (exhausted after ~25 races)
- v3: one-time cache download ≈ **few hundred units total**, then zero quota for all matching

**Daily workflow**
1. **Cell 3** — Refresh local YouTube cache (run weekly, or after new races air)
2. **Cell 5** — `run_automated_search()` — local matching, no quota cost, process all 1,000 races in one run
3. **Cell 6** — `run_transcript_fetch()` — unchanged from v2
4. **Cell 7** — Manual override (unchanged)
5. **Cell 8** — Status dashboard (unchanged)
6. **Cell 10** — Claude tactical extraction (unchanged)
7. **Cell 11** — Commentary retrieval (unchanged)


In [1]:
import os
import re
import json
import time
import random
import datetime
from pathlib import Path
from dotenv import load_dotenv

import pandas as pd
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from youtube_transcript_api import (
    YouTubeTranscriptApi,
    NoTranscriptFound,
    TranscriptsDisabled,
    VideoUnavailable,
)
from rapidfuzz import fuzz
import anthropic

load_dotenv()

YOUTUBE_API_KEY = os.getenv('YOUTUBE_API_KEY')
PROCESSED_DATA  = Path('../data/processed')
RAW_DIR         = Path('../data/commentary/raw')
EXTRACTED_DIR   = Path('../data/commentary/extracted')
CACHE_DIR       = Path('../data/commentary/cache')

RAW_DIR.mkdir(parents=True, exist_ok=True)
EXTRACTED_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

CACHE_FILE = CACHE_DIR / 'all_channel_videos.parquet'

ytt     = YouTubeTranscriptApi()
youtube = build('youtube', 'v3', developerKey=YOUTUBE_API_KEY)
claude  = anthropic.Anthropic()

print(f'YouTube API key: {"SET" if YOUTUBE_API_KEY else "MISSING"}')
print(f'Raw dir:         {RAW_DIR}')
print(f'Extracted dir:   {EXTRACTED_DIR}')
print(f'Cache dir:       {CACHE_DIR}')
print(f'Raw files:       {len(list(RAW_DIR.glob("*.json")))}')
print(f'Extracted files: {len(list(EXTRACTED_DIR.glob("*.json")))}')
print(f'Cache exists:    {CACHE_FILE.exists()}')


YouTube API key: SET
Raw dir:         ..\data\commentary\raw
Extracted dir:   ..\data\commentary\extracted
Cache dir:       ..\data\commentary\cache
Raw files:       897
Extracted files: 0
Cache exists:    True


In [2]:
CHANNEL_REGISTRY = [
    {'id': 'UCqZQlzSHbVJrwrn5XvzrzcA', 'name': 'NBC Sports',        'coverage': 'Grand Tours, Monuments, WorldTour'},
    {'id': 'UCu7phdCr-raU7OaJfEpHZww', 'name': 'GCN Racing',         'coverage': 'All WorldTour'},
    {'id': 'UCfDfvvMARk4TKcC62ALi6eA', 'name': 'TNT Sports Cycling', 'coverage': 'Grand Tours, European classics'},
    {'id': 'UCuTaETsuCOkJ0H_GAztWt0Q', 'name': 'GCN',                'coverage': 'All WorldTour'},
]
print(f'Channels: {[c["name"] for c in CHANNEL_REGISTRY]}')


Channels: ['NBC Sports', 'GCN Racing', 'TNT Sports Cycling', 'GCN']


In [3]:
# ── CELL 3: Build / refresh local YouTube cache ─────────────────────────────
# Run once to download all uploads from the four channels.
# After that, run weekly (or after a race airs) to pick up new videos.
# Quota cost: ~1–2 units per 50 videos fetched — effectively free.

def _get_upload_playlist(channel_id: str) -> str | None:
    """Get the uploads playlist ID for a channel. Costs 1 quota unit."""
    resp = youtube.channels().list(part='contentDetails', id=channel_id).execute()
    items = resp.get('items', [])
    if not items:
        return None
    return items[0]['contentDetails']['relatedPlaylists']['uploads']


def _fetch_playlist_page(playlist_id: str, page_token=None):
    """Fetch one page of playlist items. Costs 1 quota unit."""
    kwargs = dict(part='snippet', playlistId=playlist_id, maxResults=50)
    if page_token:
        kwargs['pageToken'] = page_token
    return youtube.playlistItems().list(**kwargs).execute()


def fetch_channel_videos(channel: dict, max_pages: int = 500) -> pd.DataFrame:
    """
    Download all video metadata from a channel's uploads playlist.
    max_pages=500 → up to 25,000 videos per channel.
    """
    playlist_id = _get_upload_playlist(channel['id'])
    if not playlist_id:
        print(f'  No uploads playlist found for {channel["name"]}')
        return pd.DataFrame()

    rows, page_token, pages = [], None, 0
    while pages < max_pages:
        resp       = _fetch_playlist_page(playlist_id, page_token)
        pages     += 1
        for item in resp.get('items', []):
            s = item['snippet']
            vid_id = s.get('resourceId', {}).get('videoId')
            if not vid_id or s.get('title') == 'Private video':
                continue
            rows.append({
                'video_id':  vid_id,
                'title':     s['title'],
                'published': s['publishedAt'],
                'channel':   channel['name'],
                'channel_id': channel['id'],
            })
        page_token = resp.get('nextPageToken')
        if not page_token:
            break
        time.sleep(0.1)

    df = pd.DataFrame(rows)
    if not df.empty:
        df['published'] = pd.to_datetime(df['published'], utc=True)
    return df


def build_video_cache(force_refresh: bool = False) -> pd.DataFrame:
    """
    Download all channel uploads and save to parquet.
    Safe to re-run — loads from disk unless force_refresh=True.
    """
    if CACHE_FILE.exists() and not force_refresh:
        df = pd.read_parquet(CACHE_FILE)
        age_hrs = (datetime.datetime.utcnow() -
                   datetime.datetime.fromtimestamp(CACHE_FILE.stat().st_mtime)).seconds / 3600
        print(f'Cache loaded: {len(df):,} videos (last updated {age_hrs:.1f}h ago)')
        print(f'Run build_video_cache(force_refresh=True) to re-download')
        return df

    print('Downloading channel uploads — this takes a few minutes...')
    all_dfs = []
    for ch in CHANNEL_REGISTRY:
        print(f'  {ch["name"]}...')
        ch_df = fetch_channel_videos(ch)
        print(f'    → {len(ch_df):,} videos')
        all_dfs.append(ch_df)
        time.sleep(1)

    df = pd.concat(all_dfs, ignore_index=True)
    df.to_parquet(CACHE_FILE)
    print(f'\nCache saved: {len(df):,} videos → {CACHE_FILE}')
    return df


def refresh_recent(days_back: int = 30) -> pd.DataFrame:
    """
    Lightweight refresh: re-fetch only the first few pages of each channel
    to pick up new uploads without re-downloading the entire history.
    Costs ~8 quota units total (one per channel × 2 pages).
    """
    if not CACHE_FILE.exists():
        print('No cache found — run build_video_cache() first')
        return pd.DataFrame()

    existing = pd.read_parquet(CACHE_FILE)
    cutoff   = pd.Timestamp.utcnow() - pd.Timedelta(days=days_back)
    new_rows = []

    for ch in CHANNEL_REGISTRY:
        playlist_id = _get_upload_playlist(ch['id'])
        if not playlist_id:
            continue
        page_token = None
        for _ in range(4):   # ≤4 pages = 200 newest videos per channel
            resp = _fetch_playlist_page(playlist_id, page_token)
            for item in resp.get('items', []):
                s = item['snippet']
                vid_id = s.get('resourceId', {}).get('videoId')
                if not vid_id:
                    continue
                pub = pd.to_datetime(s['publishedAt'], utc=True)
                if pub < cutoff:
                    break
                if vid_id not in existing['video_id'].values:
                    new_rows.append({'video_id': vid_id, 'title': s['title'],
                                     'published': pub, 'channel': ch['name'],
                                     'channel_id': ch['id']})
            page_token = resp.get('nextPageToken')
            if not page_token:
                break
        time.sleep(0.5)

    if new_rows:
        fresh = pd.DataFrame(new_rows)
        merged = pd.concat([existing, fresh], ignore_index=True).drop_duplicates('video_id')
        merged.to_parquet(CACHE_FILE)
        print(f'Refresh complete: added {len(new_rows)} new videos ({len(merged):,} total)')
        return merged
    else:
        print(f'Refresh complete: no new videos found')
        return existing


# ── RUN ONCE (or weekly) ──────────────────────────────────────────────────────
# First time: downloads everything (~few hundred quota units)
# After that: loads from disk instantly
all_videos = build_video_cache()


Cache loaded: 39,463 videos (last updated 21.9h ago)
Run build_video_cache(force_refresh=True) to re-download


C:\Users\Admin\AppData\Local\Temp\ipykernel_6512\992991223.py:67: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  age_hrs = (datetime.datetime.utcnow() -


In [4]:
# ── CELL 4: Helper functions + race index ────────────────────────────────────
# Identical to v2 — no changes needed here.

def parse_race_name_and_stage(race_name_full: str):
    clean       = re.sub(r'^\d{4}\s+', '', race_name_full).strip()
    stage_match = re.search(r'Stage\s+(\d+)', clean, re.IGNORECASE)
    stage       = int(stage_match.group(1)) if stage_match else None
    race        = re.sub(r'\s*Stage\s+\d+', '', clean, flags=re.IGNORECASE).strip()
    return race, stage


def make_label(race_name, race_date, stage):
    label = f'{str(race_date)[:4]} {race_name}'
    return label + (f' Stage {stage}' if stage else '')


def make_safe_name(label):
    return re.sub(r'[^a-z0-9]+', '_', label.lower()).strip('_')


def fetch_transcript(video_id: str) -> dict:
    try:
        transcript = ytt.fetch(video_id)
        raw_text   = ' '.join([s.text for s in transcript])
        clean_text = re.sub(r'\[.*?\]', '', raw_text)
        clean_text = re.sub(r'\(.*?\)', '', clean_text)
        clean_text = re.sub(r'\s+', ' ', clean_text).strip()
        duration   = round(transcript[-1].start, 0) if transcript else 0
        return {
            'success':       True,
            'video_id':      video_id,
            'snippet_count': len(transcript),
            'raw_chars':     len(raw_text),
            'clean_chars':   len(clean_text),
            'duration_mins': round(duration / 60, 1),
            'clean_text':    clean_text,
            'preview_start': clean_text[:500],
            'preview_end':   clean_text[-500:],
            'error':         None,
        }
    except NoTranscriptFound:
        return {'success': False, 'video_id': video_id, 'error': 'no_transcript'}
    except TranscriptsDisabled:
        return {'success': False, 'video_id': video_id, 'error': 'transcripts_disabled'}
    except VideoUnavailable:
        return {'success': False, 'video_id': video_id, 'error': 'video_unavailable'}
    except Exception as e:
        return {'success': False, 'video_id': video_id, 'error': str(e)}


def save_no_video(label, race_name, race_date, stage, reason='no_video_found') -> Path:
    safe_name = make_safe_name(label)
    out_path  = RAW_DIR / f'{safe_name}.json'
    result = {
        'label':      label,
        'race_name':  race_name,
        'race_date':  str(race_date)[:10],
        'stage':      stage,
        'video':      None,
        'transcript': None,
        'status':     reason,
    }
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(result, f, indent=2)
    return out_path


# Build race index — sort newest-first so recent (more likely to have transcripts) run first
merged_df = pd.read_csv(PROCESSED_DATA / 'merged_uci_races.csv', low_memory=False)
merged_df['Date'] = pd.to_datetime(merged_df['Date'])

race_index = (
    merged_df[['Race Name', 'Race_results', 'Date', 'Year_results', 'Stage_results']]
    .drop_duplicates('Race Name')
    .sort_values('Date', ascending=False)   # ← newest first (more likely to have transcripts/videos)
    .reset_index(drop=True)
)

raw_files   = list(RAW_DIR.glob('*.json'))
saved_names = {f.stem for f in raw_files}

statuses = {}
for path in raw_files:
    with open(path, encoding='utf-8') as f:
        data = json.load(f)
    s = data.get('status', 'unknown')
    statuses[s] = statuses.get(s, 0) + 1

transcript_saved = statuses.get('transcript_saved', 0)
no_video         = statuses.get('no_video_found', 0)
video_found      = statuses.get('video_found', 0)
ip_errors        = sum(v for k, v in statuses.items() if 'block' in k.lower() or 'ip' in k.lower())
other_errors     = sum(v for k, v in statuses.items()
                       if k not in ['transcript_saved', 'no_video_found', 'video_found', 'unknown']
                       and 'block' not in k.lower() and 'ip' not in k.lower())
remaining        = len(race_index) - len(raw_files)

print(f'Total races in dataset:   {len(race_index):,}')
print(f'Already processed:        {len(raw_files):,}')
print(f'  Transcripts saved:      {transcript_saved}')
print(f'  Video found (pending):  {video_found}')
print(f'  No video found:         {no_video}')
print(f'  IP/transcript errors:   {ip_errors + other_errors}')
print(f'Remaining (unprocessed):  {remaining}')
print(f'Est. time at 500/run:     {remaining // 500 + 1} runs (no quota limit!)')


Total races in dataset:   896
Already processed:        897
  Transcripts saved:      173
  Video found (pending):  264
  No video found:         454
  IP/transcript errors:   179
Remaining (unprocessed):  -1
Est. time at 500/run:     0 runs (no quota limit!)


In [5]:
to_delete = []
for path in RAW_DIR.glob('*.json'):
    with open(path, encoding='utf-8') as f:
        data = json.load(f)
    if data.get('status') == 'video_found':
        to_delete.append(path)

print(f'Will delete: {len(to_delete)} video_found records')
for path in to_delete:
    path.unlink()
print('Done')

Will delete: 264 video_found records
Done


In [6]:
# ── CELL 5: Local matching — zero quota cost ─────────────────────────────────
# Replaces the old search_channel / find_best_video API calls.
# All matching happens locally against the downloaded cache.
#
# Key improvements over v2/v3:
#   - Channel priority tiers: NBC Sports > GCN Racing > TNT/GCN
#   - Grand tour fast-path: NBC extended highlights wins automatically if score >= threshold
#   - Per-race channel preference map for known coverage patterns
#   - Title pattern boosting for known reliable title formats

SKIP_KEYWORDS = [
    'preview', 'interview', 'training', 'behind the scenes',
    'beginner', 'guide', 'how to', 'shorts', 'power data',
    'riders to watch', 'ones to watch', 'top 10',
    'what to watch', 'race preview', 'stage preview',
    'team time trial preparation', 'kitchen', 'mechanic',
    'neutral service', 'french phrases', 'injuries',
    'dutch corner', 'how to watch',
]
WOMEN_KEYWORDS = ['femmes', 'feminine', 'women', 'ladies']

# Channel priority tiers — higher = better
CHANNEL_TIER = {
    'NBC Sports':        3,
    'GCN Racing':        2,
    'TNT Sports Cycling': 1,
    'GCN':               1,
}

# Races where NBC Sports is the strongly preferred source
NBC_PRIORITY_RACES = [
    'tour de france',
    'vuelta a espana',
    'vuelta españa',
]

# Races where GCN Racing / TNT are more likely to have the best content
GCN_PRIORITY_RACES = [
    "giro d'italia",
    'giro d italia',
    'paris-roubaix',
    'paris roubaix',
    'tour of flanders',
    'ronde van vlaanderen',
    'liege-bastogne-liege',
    'milan-san remo',
    'milan san remo',
    'il lombardia',
]


def _is_nbc_priority(race_name: str) -> bool:
    rn = race_name.lower()
    return any(r in rn for r in NBC_PRIORITY_RACES)


def _is_gcn_priority(race_name: str) -> bool:
    rn = race_name.lower()
    return any(r in rn for r in GCN_PRIORITY_RACES)


def score_local_video(row: pd.Series, race_name: str, race_date: str, stage=None) -> float:
    """
    Score a candidate video row from the local cache.
    Higher score = better match.
    """
    score    = 0.0
    title    = str(row['title']).lower()
    channel  = str(row.get('channel', ''))
    race_dt  = pd.to_datetime(race_date, utc=True)
    pub_dt   = pd.to_datetime(row['published'], utc=True)
    days_diff = abs((pub_dt.date() - race_dt.date()).days)

    # ── Publish date proximity ────────────────────────────────────────────────
    if days_diff == 0:      score += 60
    elif days_diff <= 1:    score += 50
    elif days_diff <= 3:    score += 35
    elif days_diff <= 7:    score += 15
    elif days_diff > 365:   score -= 40 * abs(pub_dt.year - race_dt.year)

    # ── Race name fuzzy match ─────────────────────────────────────────────────
    name_score = fuzz.partial_ratio(race_name.lower(), title)
    score += name_score * 0.25
    if name_score < 40:     score -= 40

    # ── Stage match ──────────────────────────────────────────────────────────
    if stage:
        score += 30 if any(p in title for p in [
            f'stage {stage}', f'stage{stage}',
            f'étape {stage}', f'tappa {stage}'
        ]) else -10

    # ── Year in title ─────────────────────────────────────────────────────────
    race_year = str(race_date)[:4]
    if race_year in title:  score += 15

    # ── Content type bonuses ──────────────────────────────────────────────────
    if 'extended highlights' in title:  score += 20   # best possible cut
    elif 'extended' in title:           score += 12
    elif 'highlights' in title:         score += 5
    if 'full race' in title:            score += 8

    # ── Channel tier bonus ────────────────────────────────────────────────────
    tier = CHANNEL_TIER.get(channel, 0)
    score += tier * 5   # NBC +15, GCN Racing +10, TNT/GCN +5

    # ── NBC priority races: extra bonus for NBC extended highlights ───────────
    if _is_nbc_priority(race_name):
        if channel == 'NBC Sports':
            score += 20
            if 'extended highlights' in title:
                score += 25   # NBC extended on a grand tour = almost certainly correct

    # ── GCN priority races: boost GCN Racing / TNT ───────────────────────────
    if _is_gcn_priority(race_name):
        if channel in ('GCN Racing', 'TNT Sports Cycling'):
            score += 15

    # ── Skip penalties ────────────────────────────────────────────────────────
    if any(k in title for k in SKIP_KEYWORDS):   score -= 30   # stronger penalty than v2
    if any(k in title for k in WOMEN_KEYWORDS):  score -= 30

    return round(score, 2)


def find_best_video_local(
    race_name: str,
    race_date: str,
    stage=None,
    threshold: float = 75.0,
) -> dict:
    """
    Find the best matching video from the local cache.
    Zero quota cost. Returns same dict structure as v2's find_best_video().

    For NBC-priority grand tours, tries NBC Sports first and returns
    immediately if a strong match is found — avoids GCN junk polluting
    the candidate pool.
    """
    if all_videos is None or all_videos.empty:
        return {'found': False, 'error': 'cache_empty'}

    race_year = str(race_date)[:4]

    # Pre-filter by year
    candidates = all_videos[
        all_videos['title'].str.contains(race_year, case=False, na=False)
    ].copy()

    if candidates.empty:
        return {'found': False}

    # Further narrow by first significant race name keywords
    keywords = [w for w in race_name.split() if len(w) > 3][:2]
    if keywords:
        pattern = '|'.join(re.escape(k) for k in keywords)
        narrow  = candidates[candidates['title'].str.contains(pattern, case=False, na=False)]
        if not narrow.empty:
            candidates = narrow

    # ── NBC fast-path for grand tours ─────────────────────────────────────────
    # If this is a Tour/Vuelta, score NBC Sports candidates first.
    # If the best NBC match clears the threshold, return it immediately
    # without letting GCN kitchen-truck videos pollute the ranking.
    if _is_nbc_priority(race_name):
        nbc_candidates = candidates[candidates['channel'] == 'NBC Sports'].copy()
        if not nbc_candidates.empty:
            nbc_candidates['match_score'] = nbc_candidates.apply(
                lambda r: score_local_video(r, race_name, race_date, stage), axis=1
            )
            nbc_candidates = nbc_candidates.sort_values('match_score', ascending=False)
            nbc_best = nbc_candidates.iloc[0]
            if nbc_best['match_score'] >= threshold:
                return {
                    'found':        True,
                    'video_id':     nbc_best['video_id'],
                    'title':        nbc_best['title'],
                    'published':    str(nbc_best['published']),
                    'channel':      nbc_best['channel'],
                    'url':          f"https://youtube.com/watch?v={nbc_best['video_id']}",
                    'match_score':  float(nbc_best['match_score']),
                    'all_candidates': nbc_candidates.head(10).to_dict('records'),
                }
            # NBC had candidates but none cleared threshold — fall through to full pool
            # but keep NBC candidates in the mix

    # ── Full pool scoring ─────────────────────────────────────────────────────
    candidates['match_score'] = candidates.apply(
        lambda r: score_local_video(r, race_name, race_date, stage), axis=1
    )
    candidates = candidates.sort_values('match_score', ascending=False)
    best       = candidates.iloc[0]

    if best['match_score'] < threshold:
        return {'found': False}

    return {
        'found':        True,
        'video_id':     best['video_id'],
        'title':        best['title'],
        'published':    str(best['published']),
        'channel':      best['channel'],
        'url':          f"https://youtube.com/watch?v={best['video_id']}",
        'match_score':  float(best['match_score']),
        'all_candidates': candidates.head(10).to_dict('records'),
    }


def run_automated_search(
    max_races: int = None,
    verbose: bool = False,
    threshold: float = 75.0,
) -> dict:
    """
    Match all unprocessed races against the local video cache.
    Zero YouTube API quota — runs as fast as your CPU.
    Safe to run multiple times — skips already-processed races.
    """
    total_to_do = sum(
        1 for _, row in race_index.iterrows()
        if not (RAW_DIR / f'{make_safe_name(make_label(*parse_race_name_and_stage(row["Race Name"]), str(row["Date"])[:10]))}.json').exists()
    )
    limit = max_races if max_races is not None else total_to_do

    print(f'Local cache: {len(all_videos):,} videos across {all_videos["channel"].nunique()} channels')
    print(f'Races to process: {total_to_do:,} (limit: {limit:,})')
    print(f'{chr(8212)*55}')

    processed = skipped = found = not_found = 0
    no_video_races = []

    for i, row in race_index.iterrows():
        if processed >= limit:
            break

        race_name, stage = parse_race_name_and_stage(row['Race Name'])
        race_date        = str(row['Date'])[:10]
        label            = make_label(race_name, race_date, stage)
        safe_name        = make_safe_name(label)
        out_path         = RAW_DIR / f'{safe_name}.json'

        if out_path.exists():
            skipped += 1
            continue

        if processed % 100 == 0:
            print(f'  [{processed:>4} new | {found} found | {not_found} no_video]')

        video = find_best_video_local(race_name, race_date, stage, threshold=threshold)

        if video.get('found'):
            record = {
                'label':      label,
                'race_name':  race_name,
                'race_date':  race_date,
                'stage':      stage,
                'video': {
                    'video_id':       video['video_id'],
                    'title':          video['title'],
                    'url':            video['url'],
                    'channel':        video['channel'],
                    'published':      video['published'],
                    'match_score':    video['match_score'],
                    'all_candidates': [
                        {'video_id': c['video_id'], 'title': c['title'],
                         'channel': c['channel'], 'published': str(c['published']),
                         'match_score': c['match_score']}
                        for c in video.get('all_candidates', [])
                    ],
                },
                'transcript': None,
                'status':     'video_found',
            }
            with open(out_path, 'w', encoding='utf-8') as f:
                json.dump(record, f, indent=2, ensure_ascii=False)
            found += 1
            if verbose:
                print(f'  ✓ [{video["match_score"]:.0f}] [{video["channel"]}] {video["title"][:55]}')
        else:
            save_no_video(label, race_name, race_date, stage)
            no_video_races.append(label)
            not_found += 1

        processed += 1

    print(f'\n{chr(8212)*55}')
    print(f'Search complete (no API quota used)')
    print(f'  Videos found:    {found}')
    print(f'  No video:        {not_found}')
    print(f'  Skipped:         {skipped}')

    if no_video_races and verbose:
        print(f'\n── No video found ──')
        for lbl in no_video_races[:20]:
            print(f'  {lbl}')
        if len(no_video_races) > 20:
            print(f'  ... and {len(no_video_races) - 20} more')

    return {'found': found, 'not_found': not_found, 'skipped': skipped}


# ── RUN ───────────────────────────────────────────────────────────────────────
# First: delete existing video_found / no_video_found records you want to re-match.
# Or just run as-is — it will skip already-processed races.
search_stats = run_automated_search(
    max_races=None,
    verbose=True,
    threshold=75.0,
)

Local cache: 39,463 videos across 4 channels
Races to process: 896 (limit: 896)
———————————————————————————————————————————————————————
  [   0 new | 0 found | 0 no_video]
  ✓ [130] [GCN Racing] Superstar Climbers Clash In The Year's Final Monument |
  ✓ [137] [GCN Racing] La Vuelta Concludes With A Nail-Biting Finale! | Vuelta
  ✓ [135] [GCN Racing] A Brutal Stage Of Climbing Produces A Thrilling Finale!
  ✓ [135] [GCN Racing] A Rest For GC As The Sprinters Take Charge! | Vuelta A 
  ✓ [135] [GCN Racing] Teamwork Makes The Dream Work! | Vuelta A España 2023 H
  ✓ [135] [GCN Racing] A Jumbo Battle On The Angliru? | Vuelta A España 2023 H
  ✓ [135] [GCN Racing] Fast & Furious Racing With A Steep Climb To The Line! |
  ✓ [135] [GCN Racing] A Day Of Attrition Before The Final Rest Day! | Vuelta 
  ✓ [134] [GCN Racing] Redemption In The Pyrenees! | Vuelta A España 2023 High
  ✓ [135] [GCN Racing] The Tourmalet Takes Some Big Victims! | Vuelta A España
  ✓ [134] [GCN Racing] A Long Day Wait

In [11]:
# ── CELL 6: Fetch transcripts ────────────────────────────────────────────────
# Unchanged from v2. No API quota — just HTTP requests to YouTube.
# Uses deliberate delays to avoid IP blocking.

def run_transcript_fetch(
    max_transcripts: int = 50,
    delay_seconds: float = 45.0,
) -> dict:
    pending = []
    for path in sorted(RAW_DIR.glob('*.json')):
        with open(path, encoding='utf-8') as f:
            data = json.load(f)
        if data.get('status') == 'video_found':
            pending.append((path, data))

    total = min(len(pending), max_transcripts)
    print(f'Videos pending transcript: {len(pending)}')
    print(f'Fetching up to:            {total}')
    print(f'Delay between requests:    {delay_seconds}s')
    print(f'Estimated time:            {total * delay_seconds / 60:.1f} minutes')
    print(f'{chr(8212)*55}')

    success = errors = ip_blocked = 0

    for path, data in pending[:max_transcripts]:
        label    = data['label']
        video    = data['video']
        video_id = video['video_id']

        print(f'\nFetching: {label}')
        print(f'  {video["title"][:60]}')

        transcript = fetch_transcript(video_id)

        if transcript['success']:
            data['transcript'] = {
                'snippet_count': transcript['snippet_count'],
                'raw_chars':     transcript['raw_chars'],
                'clean_chars':   transcript['clean_chars'],
                'duration_mins': transcript['duration_mins'],
                'clean_text':    transcript['clean_text'],
                'preview_start': transcript['preview_start'],
                'preview_end':   transcript['preview_end'],
            }
            data['status'] = 'transcript_saved'
            with open(path, 'w', encoding='utf-8') as f:
                json.dump(data, f, indent=2, ensure_ascii=False)
            print(f'  ✓ {transcript["clean_chars"]:,} chars | {transcript["duration_mins"]:.1f} min')
            success += 1
        else:
            error_msg  = transcript['error']
            is_ip_block = any(kw in error_msg.lower()
                              for kw in ['blocked', 'ip', '429', 'too many'])
            if is_ip_block:
                data['status'] = 'transcript_error:ip_blocked'
                with open(path, 'w', encoding='utf-8') as f:
                    json.dump(data, f, indent=2)
                ip_blocked += 1
                print(f'\n⚠ IP blocking detected — stopping.')
                print(f'  Saved {success} transcripts before block.')
                break
            else:
                data['status'] = f'transcript_error:{error_msg[:80]}'
                with open(path, 'w', encoding='utf-8') as f:
                    json.dump(data, f, indent=2)
                errors += 1
                print(f'  ✗ {error_msg[:80]}')

        actual_delay = delay_seconds + random.uniform(-5, 10)
        actual_delay = max(20, actual_delay)
        print(f'  Waiting {actual_delay:.0f}s...')
        time.sleep(actual_delay)

    ip_blocked_total = sum(
        1 for p in RAW_DIR.glob('*.json')
        if json.load(open(p, encoding='utf-8')).get('status') == 'transcript_error:ip_blocked'
    )

    print(f'\n{chr(8212)*55}')
    print(f'Fetch complete')
    print(f'  Transcripts saved:   {success}')
    print(f'  IP blocked:          {ip_blocked}')
    print(f'  Other errors:        {errors}')
    print(f'  Still pending retry: {ip_blocked_total}')
    return {'success': success, 'ip_blocked': ip_blocked, 'errors': errors}


# ── RUN AFTER CELL 5 ─────────────────────────────────────────────────────────
fetch_stats = run_transcript_fetch(
    max_transcripts=50,
    delay_seconds=45.0,
)


Videos pending transcript: 119
Fetching up to:            50
Delay between requests:    45.0s
Estimated time:            37.5 minutes
———————————————————————————————————————————————————————

Fetching: 2022 Tour de Suisse Stage 2
  Late Attacking On A Punchy Parcours! | Tour De Suisse 2022 M

⚠ IP blocking detected — stopping.
  Saved 0 transcripts before block.

———————————————————————————————————————————————————————
Fetch complete
  Transcripts saved:   0
  IP blocked:          1
  Other errors:        0
  Still pending retry: 9


In [ ]:
# ── CELL 7: Manual override ──────────────────────────────────────────────────
# Use to: add a video you found manually, retry IP-blocked races, or fix bad matches.
# Unchanged from v2.

def show_manual_todo() -> list:
    todo = []
    for path in sorted(RAW_DIR.glob('*.json')):
        with open(path, encoding='utf-8') as f:
            data = json.load(f)
        s = data.get('status', '')
        if s in ('no_video_found',) or 'ip_blocked' in s:
            todo.append({'label': data['label'], 'status': s, 'path': path})
    print(f'Manual attention needed: {len(todo)}')
    for t in todo[:30]:
        print(f'  [{t["status"]}] {t["label"]}')
    if len(todo) > 30:
        print(f'  ... and {len(todo) - 30} more')
    return todo


def manual_add(video_id: str, label: str = None) -> dict:
    if label is None:
        print('Provide a label from show_manual_todo() output.')
        show_manual_todo()
        return {}

    safe_name = make_safe_name(label)
    out_path  = RAW_DIR / f'{safe_name}.json'

    # Load existing metadata if available
    race_name, race_date, stage = label, '2017-01-01', None
    if out_path.exists():
        with open(out_path, encoding='utf-8') as f:
            existing = json.load(f)
        race_name = existing.get('race_name', label)
        race_date = existing.get('race_date', '2017-01-01')
        stage     = existing.get('stage')
    else:
        # Try to find in race_index
        for _, row in race_index.iterrows():
            rn, st = parse_race_name_and_stage(row['Race Name'])
            rd     = str(row['Date'])[:10]
            if make_label(rn, rd, st) == label:
                race_name, race_date, stage = rn, rd, st
                break

    print(f'Fetching transcript for: {label}')
    print(f'  Video ID: {video_id}')
    transcript = fetch_transcript(video_id)

    if transcript['success']:
        record = {
            'label':     label,
            'race_name': race_name,
            'race_date': race_date,
            'stage':     stage,
            'video': {
                'video_id':    video_id,
                'title':       f'Manual add: {video_id}',
                'url':         f'https://youtube.com/watch?v={video_id}',
                'channel':     'manual',
                'published':   '',
                'match_score': 999,
            },
            'transcript': {
                'snippet_count': transcript['snippet_count'],
                'raw_chars':     transcript['raw_chars'],
                'clean_chars':   transcript['clean_chars'],
                'duration_mins': transcript['duration_mins'],
                'clean_text':    transcript['clean_text'],
                'preview_start': transcript['preview_start'],
                'preview_end':   transcript['preview_end'],
            },
            'status': 'transcript_saved',
        }
        with open(out_path, 'w', encoding='utf-8') as f:
            json.dump(record, f, indent=2, ensure_ascii=False)
        print(f'  ✓ Saved: {transcript["clean_chars"]:,} chars')
        return record
    else:
        print(f'  ✗ Failed: {transcript["error"]}')
        return {'error': transcript['error']}


def retry_ip_blocked(delay_seconds: float = 90.0) -> dict:
    blocked = []
    for path in sorted(RAW_DIR.glob('*.json')):
        with open(path, encoding='utf-8') as f:
            data = json.load(f)
        status   = data.get('status', '')
        video    = data.get('video') or {}
        video_id = video.get('video_id')
        is_blocked = any(kw in status.lower()
                         for kw in ['ip_blocked', 'blocking requests', 'too many requests'])
        if is_blocked and video_id:
            blocked.append((path, data))

    print(f'IP blocked races: {len(blocked)}')
    success = still_blocked = errors = 0

    for path, data in blocked:
        label    = data.get('label', path.stem)
        video_id = data['video']['video_id']
        print(f'\nRetrying: {label}')
        transcript = fetch_transcript(video_id)
        if transcript['success']:
            data['transcript'] = {k: transcript[k] for k in
                ['snippet_count','raw_chars','clean_chars','duration_mins',
                 'clean_text','preview_start','preview_end']}
            data['status'] = 'transcript_saved'
            with open(path, 'w', encoding='utf-8') as f:
                json.dump(data, f, indent=2, ensure_ascii=False)
            print(f'  ✓ {transcript["clean_chars"]:,} chars')
            success += 1
        else:
            err = transcript['error']
            if 'blocking' in err.lower() or 'ip' in err.lower():
                print('  ✗ Still IP blocked — stopping')
                still_blocked += 1
                break
            data['status'] = f'transcript_error:{err[:100]}'
            with open(path, 'w', encoding='utf-8') as f:
                json.dump(data, f, indent=2)
            errors += 1
            print(f'  ✗ {err[:80]}')

        actual = max(30, delay_seconds + random.uniform(-10, 15))
        print(f'  Waiting {actual:.0f}s...')
        time.sleep(actual)

    print(f'\nRetry done — saved: {success}, still blocked: {still_blocked}, errors: {errors}')
    return {'success': success, 'still_blocked': still_blocked, 'errors': errors}


# Uncomment to use:
# show_manual_todo()
# manual_add('VIDEO_ID_HERE', '2019 Paris-Roubaix')
# retry_ip_blocked()


In [ ]:
# ── CELL 8: Status dashboard ─────────────────────────────────────────────────

def show_status() -> None:
    raw_files = list(RAW_DIR.glob('*.json'))
    statuses  = {}
    for path in raw_files:
        with open(path, encoding='utf-8') as f:
            data = json.load(f)
        s = data.get('status', 'unknown')
        bucket = (s if s in ('transcript_saved', 'no_video_found', 'video_found')
                  else ('ip_blocked' if 'ip_blocked' in s else 'other_error'))
        statuses[bucket] = statuses.get(bucket, 0) + 1

    extracted_count = len(list(EXTRACTED_DIR.glob('*.json')))
    remaining       = len(race_index) - len(raw_files)
    cache_size      = len(all_videos) if all_videos is not None else 0

    print('══════ Commentary Pipeline Status ══════')
    print(f'Local cache:             {cache_size:,} videos')
    print(f'Total races:             {len(race_index):,}')
    print(f'Processed:               {len(raw_files):,}')
    print(f'  ✓ Transcript saved:    {statuses.get("transcript_saved", 0)}')
    print(f'  → Video found (queue): {statuses.get("video_found", 0)}')
    print(f'  ✗ No video:            {statuses.get("no_video_found", 0)}')
    print(f'  ⚠ IP blocked:          {statuses.get("ip_blocked", 0)}')
    print(f'  ? Other errors:        {statuses.get("other_error", 0)}')
    print(f'Remaining:               {remaining:,}')
    print(f'Extracted (Claude):      {extracted_count}')
    print()
    print(f'Next steps:')
    if statuses.get('video_found', 0) > 0:
        print(f'  → Run Cell 6 to fetch {statuses["video_found"]} pending transcripts')
    if remaining > 0:
        print(f'  → Run Cell 5 to match {remaining} unprocessed races')
    if statuses.get('ip_blocked', 0) > 0:
        print(f'  → Run retry_ip_blocked() in Cell 7')


show_status()


In [16]:
# ── CELL 9: Audit transcript quality ─────────────────────────────────────────
# Flag transcripts that look like preview/preview videos instead of race highlights.
# Unchanged from v2.

BAD_KEYWORDS = [
    'top 10 riders to watch', 'preview', 'riders to watch',
    'team time trial preparation', 'guide to', 'how to watch',
    'what to expect', 'ones to watch'
]

def audit_transcript_quality(year_filter: int = None) -> list:
    flagged = []
    for path in sorted(RAW_DIR.glob('*.json')):
        with open(path, encoding='utf-8') as f:
            data = json.load(f)
        if data.get('status') != 'transcript_saved':
            continue
        label = data.get('label', '')
        if year_filter and not label.startswith(str(year_filter)):
            continue
        title = data.get('video', {}).get('title', '').lower()
        if any(kw in title for kw in BAD_KEYWORDS):
            flagged.append({'label': label, 'title': data['video']['title'], 'path': path})

    print(f'Flagged as likely bad matches: {len(flagged)}')
    for f in flagged:
        print(f'  {f["label"]}')
        print(f'    {f["title"]}')
    return flagged


def delete_bad_matches(flagged: list, dry_run: bool = True) -> None:
    for f in flagged:
        if dry_run:
            print(f'Would reset: {f["path"].name}')
        else:
            data  = json.load(open(f['path'], encoding='utf-8'))
            clean = {
                'label':      data['label'],
                'race_name':  data.get('race_name', ''),
                'race_date':  data.get('race_date', ''),
                'stage':      data.get('stage'),
                'video':      None,
                'transcript': None,
                'status':     'no_video_found',
            }
            with open(f['path'], 'w', encoding='utf-8') as out:
                json.dump(clean, out, indent=2)
            print(f'Reset: {f["path"].name}')


bad_matches = audit_transcript_quality()
delete_bad_matches(bad_matches, dry_run=True)   # preview
# delete_bad_matches(bad_matches, dry_run=False)  # execute


Flagged as likely bad matches: 1
  2018 Vuelta a Espana Stage 1
    Vuelta a España Stage 1 Time Trial Course Preview | Vuelta a España 2018
Would reset: 2018_vuelta_a_espana_stage_1.json


In [ ]:
# ── CELL 10: Claude tactical extraction ──────────────────────────────────────
# Reads saved transcripts and extracts structured tactical events.
# Unchanged from v2.

EXTRACTION_PROMPT = """You are analyzing cycling race commentary for {race_name}.

Extract tactically significant information from this transcript.
Focus only on information useful for pre-race analysis of future similar stages.
Ignore music, crowd noise, and generic excitement phrases.

Return a JSON object:
{{
  "race_summary": "2-3 sentence tactical summary of what happened",
  "decisive_moment": "the single most important tactical event",
  "commentary_quality": "rich|moderate|thin",
  "events": [
    {{
      "event_type": "attack|breakaway|chase|team_tactic|rider_observation|weather|terrain",
      "riders": ["LASTNAME Firstname"],
      "teams": ["Team Name"],
      "description": "what happened concisely",
      "tactical_significance": "why this matters for future analysis"
    }}
  ],
  "rider_form_signals": [
    {{
      "rider": "LASTNAME Firstname",
      "signal": "positive|negative|neutral",
      "observation": "what commentary reveals about their condition"
    }}
  ],
  "terrain_observations": [
    "observation about how terrain affected racing dynamics"
  ],
  "usable_for_rag": true
}}

Return only valid JSON, no other text.
Transcript:
{transcript}"""


def extract_one(label: str, transcript_text: str) -> dict:
    max_chars = 8000
    if len(transcript_text) > max_chars:
        half = max_chars // 2
        text = transcript_text[:half] + '\n[...middle omitted...]\n' + transcript_text[-half:]
    else:
        text = transcript_text

    response = claude.messages.create(
        model='claude-sonnet-4-5',
        max_tokens=3000,
        messages=[{'role': 'user', 'content': EXTRACTION_PROMPT.format(
            race_name=label, transcript=text
        )}]
    )
    raw = re.sub(r'```json|```', '', response.content[0].text).strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {'error': 'json_parse_failed', 'raw': raw[:200]}


def run_extraction(max_extractions: int = 50, verbose: bool = True) -> dict:
    pending = [
        path for path in sorted(RAW_DIR.glob('*.json'))
        if not (EXTRACTED_DIR / path.name).exists()
        and json.load(open(path, encoding='utf-8')).get('status') == 'transcript_saved'
    ]

    total_cost = success = skipped = errors = 0
    print(f'Transcripts pending extraction: {len(pending)}')
    print(f'Processing up to: {max_extractions}')
    print(f'Est. cost: ${min(len(pending), max_extractions) * 0.02:.2f}')
    print(f'{chr(8212)*55}')

    for path in pending[:max_extractions]:
        extracted_path = EXTRACTED_DIR / path.name
        with open(path, encoding='utf-8') as f:
            data = json.load(f)
        transcript_text = data.get('transcript', {}).get('clean_text', '')
        if not transcript_text.strip():
            skipped += 1
            continue

        label = data.get('label', path.stem)
        chars = len(transcript_text)
        cost  = min(chars, 8000) / 4 / 1_000_000 * 3

        if verbose:
            print(f'Extracting: {label}')
            print(f'  Chars: {chars:,} | Est: ${cost:.4f}')

        try:
            events = extract_one(label, transcript_text)
            output = {
                'label':       label,
                'race_name':   data.get('race_name'),
                'race_date':   data.get('race_date'),
                'stage':       data.get('stage'),
                'video_title': data.get('video', {}).get('title'),
                'video_url':   data.get('video', {}).get('url'),
                'channel':     data.get('video', {}).get('channel'),
                'extraction':  events,
                'extracted_at': str(datetime.datetime.utcnow()),
            }
            with open(extracted_path, 'w', encoding='utf-8') as f:
                json.dump(output, f, indent=2, ensure_ascii=False)

            total_cost += cost
            success    += 1
            if verbose:
                quality  = events.get('commentary_quality', 'unknown')
                n_events = len(events.get('events', []))
                print(f'  ✓ Quality: {quality} | Events: {n_events}')
        except Exception as e:
            print(f'  ✗ Error: {e}')
            errors += 1

        time.sleep(0.5)

    print(f'\n{chr(8212)*55}')
    print(f'Extraction complete')
    print(f'  Extracted:   {success}')
    print(f'  Skipped:     {skipped}')
    print(f'  Errors:      {errors}')
    print(f'  Total cost:  ${total_cost:.4f}')
    return {'success': success, 'skipped': skipped, 'errors': errors, 'cost': total_cost}


# ── RUN EXTRACTION ─────────────────────────────────────────────────────────────
# extraction_stats = run_extraction(max_extractions=50, verbose=True)
print('Extraction functions ready — uncomment the line above to run')


In [ ]:
# ── CELL 11: Commentary retrieval ────────────────────────────────────────────
# Used by the agent notebook. Always returns a string — never raises.
# Unchanged from v2.

def get_commentary_context(
    race_name: str,
    race_date: str,
    stage: int = None,
    max_chars: int = 3000,
) -> str:
    label     = make_label(race_name, race_date, stage)
    safe_name = make_safe_name(label)

    extracted_path = EXTRACTED_DIR / f'{safe_name}.json'
    raw_path       = RAW_DIR       / f'{safe_name}.json'

    if extracted_path.exists():
        try:
            with open(extracted_path, encoding='utf-8') as f:
                data = json.load(f)
            ext = data.get('extraction', {})
            if 'error' not in ext:
                parts = [f'[COMMENTARY: {data.get("channel")} | {data.get("video_title","")[:55]}]']
                if ext.get('race_summary'):
                    parts.append(f'Summary: {ext["race_summary"]}')
                if ext.get('decisive_moment'):
                    parts.append(f'Decisive moment: {ext["decisive_moment"]}')
                for e in ext.get('events', [])[:5]:
                    riders = ', '.join(e.get('riders', []))
                    parts.append(f'- [{e["event_type"]}] {e["description"]}' +
                                 (f' ({riders})' if riders else ''))
                for s in ext.get('rider_form_signals', [])[:3]:
                    parts.append(f'- Form: {s["rider"]} ({s["signal"]}): {s["observation"]}')
                return '\n'.join(parts)
        except Exception:
            pass

    if raw_path.exists():
        try:
            with open(raw_path, encoding='utf-8') as f:
                data = json.load(f)
            if data.get('status') == 'transcript_saved':
                text = data.get('transcript', {}).get('clean_text', '')
                if text:
                    if len(text) > max_chars:
                        half = max_chars // 2
                        text = text[:half] + '\n[...]\n' + text[-half:]
                    channel = data.get('video', {}).get('channel', 'unknown')
                    title   = data.get('video', {}).get('title', '')[:55]
                    return f'[RAW COMMENTARY: {channel} | {title}]\n\n{text}'
        except Exception:
            pass

    return f'[NO COMMENTARY for {label}] Analysis based on structured data only.'


# Quick test
print('=== Commentary retrieval ready ===')
